# GPU Metrics Data Processing & Imputation Notebook

This notebook performs end-to-end processing of GPU utilization metrics from Nessie/Iceberg tables using Spark on Enterprise Gateway (SparkCaster). It includes feature engineering, ML-based imputation of missing `redfish_power` values, and writes the results back to Nessie.

---
#sparkcaster using "http://spark-gateway.kubeflow.svc.cluster.local:8888"

## 📋 Notebook Structure

### **Setup & Initialization** (Cells 1-5)

- **Cell 1**: Package installation - Installs required Python packages including PySpark, Nessie/Iceberg connectors, visualization libraries, and ML tools
- **Cell 2**: Credential management - Sets up encrypted keyring for secure storage of Nessie credentials
- **Cell 3**: Credential validation - Verifies Nessie connection and authentication
- **Cell 4**: Spark cluster setup - Configures SparkCaster session with Nessie/Iceberg catalog, cleans up existing sessions, and initializes Spark context
- **Cell 5**: Data loading - Queries source data from Nessie using Spark SQL

### **Feature Engineering** (Cells 6-7)

- **Cell 6**: Feature engineering pipeline - Creates derived features for modeling and analysis:
  - `tensor_to_gpu_ratio` - Tensor utilization relative to GPU utilization
  - `memory_intensity` - Memory copy utilization relative to GPU utilization
  - `compute_to_memory_ratio` - Compute throughput relative to memory usage
  - `power_%_max` - Power usage as percentage of maximum
  - `compute_%_max` - Compute usage as percentage of maximum
  - `gen` - GPU generation classification (current_gen vs prior_gen)
  - `training_inference` - Simple workload classification:
    - **active_training**: `is_training == "slurm"`
    - **non_training**: `is_training != "slurm"`
    
- **Cell 7**: DataFrame creation - Prepares final DataFrame with all features for modeling

### **ML Model Training & Evaluation** (Cells 8-12)

- **Cell 8**: Complete ML pipeline with 5-fold cross-validation - Trains and evaluates multiple regression models to predict `redfish_power`:
  - Model 1: Simple linear regression (`chip_power` only)
  - Model 2: Multiple features (`chip_power`, `gpu_util`, `tensor_util`, `tensor_tflops`)
  - Uses cross-validation to select best model
  - Caches training data for performance
  
- **Cell 9**: Model 1 training - Simple linear regression baseline using only `chip_power`
- **Cell 10**: Diagnostic plots - Generates QQ plots and predicted vs actual visualizations for all trained models
- **Cell 11**: Memory cleanup - Unpersists cached data to free memory
- **Cell 12**: Final model training - Retrains the best performing model on optimized sample size (10% of data) for production use

### **Data Imputation & Output** (Cells 13-15)

- **Cell 13**: Imputation execution - Applies the trained model to predict missing `redfish_power` values across the full dataset
  - Identifies rows with missing values
  - Applies ML model predictions
  - Creates `redfish_power_imputed` flag for tracking
  - Combines imputed and original data
  - Reports imputation statistics
  
- **Cell 14**: Schema verification - Prints final DataFrame schema after imputation

- **Cell 15**: Write to Nessie - Writes the complete imputed dataset to Nessie/Iceberg table
  - **Target table**: `sandbox.sandbox_finance.dcgm_metrics_raw_impute`
  - Includes all original columns plus engineered features
  - Uses Iceberg table format for time-travel and ACID properties
  - Reports write statistics (row count, duration, throughput)

### **Analysis & Visualization** (Cells 16-17)

- **Cell 16**: Classification analysis - Analyzes the `training_inference` feature distribution and relationships with power metrics
- **Cell 17**: Classification visualization - Creates plots showing training vs inference workload patterns

### **Data Validation & Testing** (Cells 18+)

- **Cells 18+**: Data validation and testing - Verification queries to confirm successful write and data quality

---

## 🎯 Key Features

- **Feature Engineering**: Creates 7 derived features including GPU generation, workload classification, and performance ratios
- **Simplified Classification**: Binary workload classification (active_training vs non_training) based on Slurm job detection
- **ML-based Imputation**: Uses Spark MLlib with cross-validation to select best regression model for missing power values
- **Scalable Processing**: Designed for large datasets using Spark 3.5.5 on Enterprise Gateway (SparkCaster)
- **Nessie/Iceberg Output**: Writes to Iceberg tables for versioning, time-travel, and ACID guarantees
- **Secure Credentials**: Encrypted keyring storage for database passwords
- **Data Validation**: Built-in schema verification and row count checks

---

## 🚀 Quick Start

1. **Setup** (Cells 1-4): Install packages, configure credentials, initialize SparkCaster
2. **Load Data** (Cell 5): Query source data from Nessie
3. **Feature Engineering** (Cells 6-7): Create derived features including simplified training/inference classification
4. **Train Models** (Cells 8-12): Train ML models with cross-validation, select best model
5. **Impute** (Cells 13-14): Apply model to predict missing `redfish_power` values
6. **Write Output** (Cell 15): Save imputed dataset to Nessie/Iceberg table
7. **Analyze** (Cells 16-17): Explore workload classification patterns
8. **Validate** (Cells 18+): Verify data quality and row counts

---

## 📊 Key Data Assets

### Input Data
- **Source**: Nessie/Iceberg table via Spark SQL
- **Catalog**: `sandbox.sandbox_finance`
- **Processing**: In-memory Spark DataFrames with distributed processing

### Output Data
- **Target Table**: `sandbox.sandbox_finance.dcgm_metrics_raw_impute`
- **Format**: Apache Iceberg (enables time-travel, schema evolution, ACID transactions)
- **Features**: 21+ columns including:
  - Original metrics: `datestamp`, `node`, `gpu_util`, `tensor_util`, `chip_power`, `redfish_power`
  - Engineered features: `tensor_to_gpu_ratio`, `memory_intensity`, `compute_to_memory_ratio`, `power_%_max`, `compute_%_max`, `gen`, `training_inference`
  - Imputation metadata: `redfish_power_imputed` flag

### ML Features for Imputation
Primary model predictors (selected via cross-validation):
- `chip_power` - GPU chip power consumption (primary predictor)
- `gpu_util` - GPU utilization percentage
- `tensor_util` - Tensor core utilization
- `tensor_tflops` - Tensor FLOPS performance

### Workload Classification Logic
Simple binary classification for `training_inference` feature:
- **active_training**: Records where `is_training == "slurm"` (GPU actively running Slurm training jobs)
- **non_training**: All other records (`is_training != "slurm"`)

---

## 🔧 Technical Configuration

### SparkCaster / Enterprise Gateway
- **Endpoint**: `http://spark-gateway.kubeflow.svc.cluster.local:8888`
- **Spark Version**: 3.5.5
- **Catalog**: Nessie with Iceberg tables
- **Processing**: Distributed Spark DataFrames
- **Caching**: Strategic caching for model training performance

### ML Configuration
- **Framework**: Spark MLlib
- **Validation**: 5-fold cross-validation
- **Models Tested**: Simple and multi-feature linear regression
- **Training Sample**: 10% of data (optimized for performance vs accuracy)
- **Evaluation Metrics**: RMSE, R², MAE

### Data Output
- **Format**: Apache Iceberg (via Nessie catalog)
- **Write Mode**: Append/Overwrite (configurable)
- **Validation**: Row count verification after write
- **Benefits**: Time-travel, schema evolution, ACID properties, efficient updates

---

## ⚠️ Important Notes

- Feature engineering creates the simplified `training_inference` binary classification (active_training / non_training)
- The notebook uses cross-validation to automatically select the best performing ML model
- All imputed `redfish_power` values are flagged with `redfish_power_imputed` for traceability
- Nessie/Iceberg provides versioning - you can query previous versions of the table using time-travel
- Strategic data caching is used during model training to improve performance
- The complete pipeline is designed to run end-to-end without manual intervention

In [2]:
#install needed packages

import importlib
import subprocess
import sys

def is_module_available(module_name: str) -> bool:
    """Return True if the importable module exists; handles dotted names safely."""
    try:
        return importlib.util.find_spec(module_name) is not None
    except ModuleNotFoundError:
        return False

def install_if_missing(pip_name: str, import_name: str = None):
    """
    Install `pip_name` only if `import_name` (or `pip_name` if None) is not importable.
    Use this to handle cases where pip/distribution names differ from import names.
    """
    import_name = import_name or pip_name
    if not is_module_available(import_name):
        print(f"Installing {pip_name} ...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
    else:
        print(f"{pip_name} already installed.")

# --- Packages for Nessie/Iceberg access ---
# simple 1:1 cases
install_if_missing("keyring", "keyring")
install_if_missing("ipython-secrets", "ipython_secrets")
install_if_missing("pyarrow", "pyarrow")
install_if_missing("fsspec", "fsspec")
install_if_missing("s3fs", "s3fs")
install_if_missing("statsmodels", "statsmodels")
install_if_missing("matplotlib", "matplotlib")
install_if_missing("scikit-learn", "sklearn")
install_if_missing("xgboost", "xgboost")

# special cases
# keyrings.cryptfile installs a plugin under the 'keyrings' package; checking the root avoids ModuleNotFoundError
install_if_missing("keyrings.cryptfile", "keyrings")

# Core viz + scaling pins for Python 3.11.11
install_if_missing("bokeh==3.6.2", "bokeh")
install_if_missing("jupyter_bokeh", "jupyter_bokeh")
install_if_missing("panel==1.5.2", "panel")
install_if_missing("holoviews==1.19.0", "holoviews")
install_if_missing("hvplot==0.10.0", "hvplot")
install_if_missing("datashader==0.16.3", "datashader")
install_if_missing("dask[dataframe]==2024.9.1", "dask")
install_if_missing("distributed==2024.9.1", "distributed")
install_if_missing("reportlab", "reportlab")


print("\n✓ All packages installed/verified")




keyring already installed.
ipython-secrets already installed.
pyarrow already installed.
fsspec already installed.
s3fs already installed.
Installing statsmodels ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 40.0 MB/s  0:00:00ta 0:00:02
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [statsmodels] [statsmodels]
Installing matplotlib ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 8.8/8.8 MB 39.9 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 85.0 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 252.5 MB/s  0:00:00
   ━━━━━━━━╺━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1/5 [fonttools]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5/5 [matplotlib]5 [matplotlib]
scikit-learn already installed.
Installing xgboost ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 131.7/131.7 MB 97.2 MB/s  0:00:01m0:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 303.4/303.4 MB 138.9 MB/s  0:00:020:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [xgboost]m1/2 [xgboost]
keyrings.cryptfile already installed.
Installing bokeh==3.6.2 ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 30.0 MB/s  0:00:00eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 1/2 [bokeh]

  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [bokeh]
Installing jupyter_bokeh ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.5 MB/s  0:00:00 eta 0:00:01
Installing panel==1.5.2 ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.4/27.4 MB 67.8 MB/s  0:00:00 eta 0:00:01
   ━━━━━━━━━━━━━━━━━━━━╺━━━━━━━━━━━━━━━━━━━ 3/6 [bleach]

...

Defaulting to user installation because normal site-packages is not writeable
INFO: pip is looking at multiple versions of dask-expr to determine which version is compatible with other requirements. This could take a while.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.3/1.3 MB 8.8 MB/s  0:00:00 eta 0:00:01
  Attempting uninstall: dask
    Found existing installation: dask 2026.7.1
    Uninstalling dask-2026.7.1:
      Successfully uninstalled dask-2026.7.1
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2/2 [dask-expr]


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Installing distributed==2024.9.1 ...
Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 8.8 MB/s  0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3/3 [distributed] [distributed]
Installing reportlab ...


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.


Defaulting to user installation because normal site-packages is not writeable
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 14.6 MB/s  0:00:00eta 0:00:01

✓ All packages installed/verified


In [3]:
#set up caios credentials
import keyring
import os
from getpass import getpass
from keyrings.cryptfile.cryptfile import CryptFileKeyring
from pathlib import Path

# Use home directory dynamically instead of hardcoded path
home_dir = str(Path.home())
os.environ["KEYRING_CRYPTFILE_PATH"] = f"{home_dir}/.local/share/python_keyring/cryptfile_pass.cfg"

kr = CryptFileKeyring()
kr.keyring_key = getpass("Set/enter master password for encrypted keyring: ")
keyring.set_keyring(kr)

# Prompt for CAIOS credentials if not already stored
caios_access_key = keyring.get_password("caios", "access_key")
caios_secret_key = keyring.get_password("caios", "secret_key")  

if not caios_access_key:
    caios_access_key = input("Enter CAIOS access key: ")
    keyring.set_password("caios", "access_key", caios_access_key)

if not caios_secret_key:
    caios_secret_key = getpass("Enter CAIOS secret key: ")
    keyring.set_password("caios", "secret_key", caios_secret_key)

print("✓ CAIOS credentials configured")

# ========================================
# StarRocks Credentials Setup
# ========================================
print("="*60)
print("StarRocks Credentials Setup")
print("="*60)

# Prompt for StarRocks credentials if not already stored
starrocks_username = keyring.get_password("starrocks", "username")
starrocks_password = keyring.get_password("starrocks", "password")

if not starrocks_username:
    starrocks_username = input("Enter StarRocks username: ")
    keyring.set_password("starrocks", "username", starrocks_username)

if not starrocks_password:
    starrocks_password = getpass("Enter StarRocks password: ")
    keyring.set_password("starrocks", "password", starrocks_password)

print("✓ StarRocks credentials configured")


✓ CAIOS credentials configured
StarRocks Credentials Setup
✓ StarRocks credentials configured


In [4]:
#credential validation
import keyring

# Retrieve CAIOS credentials from keyring
caios_access_key = keyring.get_password("caios", "access_key")
caios_secret_key = keyring.get_password("caios", "secret_key")

assert caios_access_key and caios_secret_key, "No CAIOS credentials in keyring."

print("✓ CAIOS credentials validated")
print(f"  Access key: {caios_access_key[:3]}...{caios_access_key[-3:]}")

✓ CAIOS credentials validated
  Access key: CWO...SKY


In [5]:
# CLEAN UP EXISTING SESSION
# ========================================
from spark.nessie.client import NessieSparkClient


try:
    existing_spark = SparkSession.getActiveSession()
    if existing_spark:
        print("Stopping existing Spark session...")
        existing_spark.stop()
        print("✓ Existing Spark session stopped")
    else:
        print("No existing Spark session found")
except Exception as e:
    print(f"Note: Could not check for existing session: {e}")

ness = NessieSparkClient(
    svc_url="http://kf-proxy.nessie.svc.cluster.local:19120/api/v2",
    nessie_endpoint="http://nessie-prd.cwobject.com",
    caios_access_key=caios_access_key,
    caios_secret_key=caios_secret_key,
    dbtcaster=True,
)

spark = ness.spark

spark.sparkContext.setLogLevel("ERROR")

# ========================================
# SESSION DIAGNOSTICS
# ========================================
print("=" * 60)
print("SPARK CLUSTER CONFIGURATION - NESSIE SPARK CLIENT")
print("=" * 60)
print(f"Spark Version: {spark.version}")
print(f"Spark Master: {spark.sparkContext.master}")
print(f"App Name: {spark.sparkContext.appName}")
print(f"Application ID: {spark.sparkContext.applicationId}")
print("Nessie Service URL: http://kf-proxy.nessie.svc.cluster.local:19120/api/v2")
print("Nessie Endpoint: http://nessie-prd.cwobject.com")
print("DBTCaster Mode: Enabled")
print("AWS Environment Variables: Set (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)")

for k in [
    "spark.driver.memory",
    "spark.driver.cores",
    "spark.executor.instances",
    "spark.executor.cores",
    "spark.executor.memory",
    "spark.sql.shuffle.partitions",
    "spark.sql.adaptive.enabled",
]:
    try:
        print(f"{k}: {spark.conf.get(k)}")
    except Exception:
        print(f"{k}: <unset>")

try:
    executor_count = spark.sparkContext._jsc.sc().getExecutorMemoryStatus().size()
    print(f"Executor entries visible: {executor_count}")
except Exception as e:
    print(f"Could not inspect executor status: {e}")

print("=" * 60)



Note: Could not check for existing session: name 'SparkSession' is not defined
SPARK CLUSTER CONFIGURATION - NESSIE SPARK CLIENT
Spark Version: 4.1.2
Spark Master: k8s://https://10.16.0.1:443
App Name: SparkClient for Nessie Catalog (and Iceberg)
Application ID: spark-53c3d228336a49ef817d4d232709c01c
Nessie Service URL: http://kf-proxy.nessie.svc.cluster.local:19120/api/v2
Nessie Endpoint: http://nessie-prd.cwobject.com
DBTCaster Mode: Enabled
AWS Environment Variables: Set (AWS_ACCESS_KEY_ID, AWS_SECRET_ACCESS_KEY)
spark.driver.memory: 16g
spark.driver.cores: 4
spark.executor.instances: 10
spark.executor.cores: 4
spark.executor.memory: 64g
spark.sql.shuffle.partitions: 200
spark.sql.adaptive.enabled: true
Executor entries visible: 9


In [7]:
sql_test='''
CREATE OR REPLACE VIEW sandbox.sandbox_jbok.v_dcgm_metrics_summary_imputed_daily

 (
    day COMMENT 'UTC day for the aggregated metrics record at daily grain.',
    region COMMENT 'Datacenter region associated with the underlying GPU activity.',
    is_training COMMENT 'Workload flag indicating whether the usage is classified as a training workload.',
    is_multinode COMMENT 'Workload flag indicating whether the job is classified as multinode.',
    product_segment COMMENT 'Hardware form-factor segment for the resolved GPU product, such as HGX, PCIE, MGX, or Workstation.',
    gpu_count_expected COMMENT 'Expected number of GPUs per node for the resolved product configuration.',
    customer_segment COMMENT 'Customer segmentation bucket used for finance and workload analysis.',
    customer_name COMMENT 'Customer name associated with the usage record.',
    region_rollup COMMENT 'Higher-level regional grouping used for reporting and aggregation.',
    zone COMMENT 'Availability zone associated with the underlying infrastructure.',
    gen COMMENT 'GPU generation or hardware generation classification.',
    product_resolved COMMENT 'Final resolved product name after enrichment and imputation logic.',
    peak_power_unit COMMENT 'Peak thermal design power in watts for a single GPU unit of the resolved product.',
    gpu_util_p50 COMMENT '50th percentile GPU utilization across the day, expressed on a 0 to 1 scale.',
    gpu_util_p95 COMMENT '95th percentile GPU utilization across the day, expressed on a 0 to 1 scale.',
    gpu_util_p99 COMMENT '99th percentile GPU utilization across the day, expressed on a 0 to 1 scale.',
    tensor_util_p50 COMMENT '50th percentile tensor core utilization across the day, expressed on a 0 to 1 scale.',
    tensor_util_p95 COMMENT '95th percentile tensor core utilization across the day, expressed on a 0 to 1 scale.',
    tensor_util_p99 COMMENT '99th percentile tensor core utilization across the day, expressed on a 0 to 1 scale.',
    chip_power_fleet_p50 COMMENT '50th percentile GPU chip power draw in watts across the day.',
    chip_power_fleet_p95 COMMENT '95th percentile GPU chip power draw in watts across the day.',
    chip_power_fleet_p99 COMMENT '99th percentile GPU chip power draw in watts across the day.',
    redfish_power_fleet_p50 COMMENT '50th percentile Redfish chassis power reading in watts across the day.',
    redfish_power_fleet_p95 COMMENT '95th percentile Redfish chassis power reading in watts across the day.',
    redfish_power_fleet_p99 COMMENT '99th percentile Redfish chassis power reading in watts across the day.',
    dram_active_p50 COMMENT '50th percentile DRAM active ratio across the day, expressed on a 0 to 1 scale.',
    dram_active_p95 COMMENT '95th percentile DRAM active ratio across the day, expressed on a 0 to 1 scale.',
    dram_active_p99 COMMENT '99th percentile DRAM active ratio across the day, expressed on a 0 to 1 scale.',
    mem_copy_util_p50 COMMENT '50th percentile memory copy utilization across the day, expressed on a 0 to 1 scale.',
    mem_copy_util_p95 COMMENT '95th percentile memory copy utilization across the day, expressed on a 0 to 1 scale.',
    mem_copy_util_p99 COMMENT '99th percentile memory copy utilization across the day, expressed on a 0 to 1 scale.',
    vram_usage_fleet_p50 COMMENT '50th percentile VRAM usage across the day, measured in MB.',
    vram_usage_fleet_p95 COMMENT '95th percentile VRAM usage across the day, measured in MB.',
    vram_usage_fleet_p99 COMMENT '99th percentile VRAM usage across the day, measured in MB.',
    sm_active_p50 COMMENT '50th percentile SM active ratio across the day, expressed on a 0 to 1 scale.',
    sm_active_p95 COMMENT '95th percentile SM active ratio across the day, expressed on a 0 to 1 scale.',
    sm_active_p99 COMMENT '99th percentile SM active ratio across the day, expressed on a 0 to 1 scale.',
    sm_clock_p50 COMMENT '50th percentile SM clock speed across the day, measured in MHz.',
    sm_clock_p95 COMMENT '95th percentile SM clock speed across the day, measured in MHz.',
    sm_clock_p99 COMMENT '99th percentile SM clock speed across the day, measured in MHz.',
    sm_occupancy_p50 COMMENT '50th percentile SM occupancy ratio across the day, expressed on a 0 to 1 scale.',
    sm_occupancy_p95 COMMENT '95th percentile SM occupancy ratio across the day, expressed on a 0 to 1 scale.',
    sm_occupancy_p99 COMMENT '99th percentile SM occupancy ratio across the day, expressed on a 0 to 1 scale.',
    memory_boundness_index_p50 COMMENT '50th percentile memory-boundness index across the day; higher values indicate relatively more memory-bound behavior.',
    memory_boundness_index_p95 COMMENT '95th percentile memory-boundness index across the day; higher values indicate relatively more memory-bound behavior.',
    memory_boundness_index_p99 COMMENT '99th percentile memory-boundness index across the day; higher values indicate relatively more memory-bound behavior.',
    TFLOPS_per_watt_efficiency_p50 COMMENT '50th percentile estimated compute-per-watt efficiency across the day, measured as TFLOPS per watt.',
    TFLOPS_per_watt_efficiency_p95 COMMENT '95th percentile estimated compute-per-watt efficiency across the day, measured as TFLOPS per watt.',
    TFLOPS_per_watt_efficiency_p99 COMMENT '99th percentile estimated compute-per-watt efficiency across the day, measured as TFLOPS per watt.',
    compute_occupancy_index_p50 COMMENT '50th percentile compute occupancy index across the day, reflecting how fully compute resources are occupied when active.',
    compute_occupancy_index_p95 COMMENT '95th percentile compute occupancy index across the day, reflecting how fully compute resources are occupied when active.',
    compute_occupancy_index_p99 COMMENT '99th percentile compute occupancy index across the day, reflecting how fully compute resources are occupied when active.',
    tensor_util_dram_index_p50 COMMENT '50th percentile tensor-utilization-to-DRAM index across the day, used as an alternative compute-versus-memory intensity indicator.',
    tensor_util_dram_index_p95 COMMENT '95th percentile tensor-utilization-to-DRAM index across the day, used as an alternative compute-versus-memory intensity indicator.',
    tensor_util_dram_index_p99 COMMENT '99th percentile tensor-utilization-to-DRAM index across the day, used as an alternative compute-versus-memory intensity indicator.',
    tensor_tflops_sm_fleet_p50 COMMENT '50th percentile estimated tensor TFLOPS adjusted for SM clock behavior across the day.',
    tensor_tflops_sm_fleet_p95 COMMENT '95th percentile estimated tensor TFLOPS adjusted for SM clock behavior across the day.',
    tensor_tflops_sm_fleet_p99 COMMENT '99th percentile estimated tensor TFLOPS adjusted for SM clock behavior across the day.',
    tensor_to_gpu_ratio_p50 COMMENT '50th percentile ratio of tensor utilization relative to overall GPU utilization across the day.',
    tensor_to_gpu_ratio_p95 COMMENT '95th percentile ratio of tensor utilization relative to overall GPU utilization across the day.',
    tensor_to_gpu_ratio_p99 COMMENT '99th percentile ratio of tensor utilization relative to overall GPU utilization across the day.',
    memory_intensity_p50 COMMENT '50th percentile memory-intensity indicator across the day.',
    memory_intensity_p95 COMMENT '95th percentile memory-intensity indicator across the day.',
    memory_intensity_p99 COMMENT '99th percentile memory-intensity indicator across the day.',
    power_pct_max_p50 COMMENT '50th percentile realized GPU power as a percentage of peak unit power across the day.',
    power_pct_max_p95 COMMENT '95th percentile realized GPU power as a percentage of peak unit power across the day.',
    power_pct_max_p99 COMMENT '99th percentile realized GPU power as a percentage of peak unit power across the day.',
    compute_to_memory_ratio_p50 COMMENT '50th percentile ratio of compute-oriented activity relative to memory-oriented activity across the day.',
    compute_to_memory_ratio_p95 COMMENT '95th percentile ratio of compute-oriented activity relative to memory-oriented activity across the day.',
    compute_to_memory_ratio_p99 COMMENT '99th percentile ratio of compute-oriented activity relative to memory-oriented activity across the day.',
    compute_pct_max_p50 COMMENT '50th percentile realized compute throughput as a percentage of theoretical maximum across the day.',
    compute_pct_max_p95 COMMENT '95th percentile realized compute throughput as a percentage of theoretical maximum across the day.',
    compute_pct_max_p99 COMMENT '99th percentile realized compute throughput as a percentage of theoretical maximum across the day.',
    node_count_daily_avg COMMENT 'Average daily node count contributing to the aggregated metrics record.',
    tflops_avg COMMENT 'Average estimated TFLOPS across contributing observations for the day.',
    tflops_total_p50 COMMENT '50th percentile total estimated TFLOPS across the day.',
    tflops_total_p95 COMMENT '95th percentile total estimated TFLOPS across the day.',
    tflops_total_p99 COMMENT '99th percentile total estimated TFLOPS across the day.',
    tflops_node_avg_p50 COMMENT '50th percentile average estimated TFLOPS per node across the day.',
    tflops_node_avg_p95 COMMENT '95th percentile average estimated TFLOPS per node across the day.',
    tflops_node_avg_p99 COMMENT '99th percentile average estimated TFLOPS per node across the day.'
)
COMMENT 'Maintained by the Capacity Financial Planning and Analysis team. Provides a daily summarized view of imputed DCGM GPU utilization, power, throughput, and workload-classification metrics by customer, product, and region for finance-oriented reporting and analysis.'

AS
SELECT
    dcgm_daily.day,
    dcgm_daily.region,
    dcgm_daily.is_training,
    dcgm_daily.is_multinode,
    dcgm_daily.product_segment,
    dcgm_daily.gpu_count_expected,
    dcgm_daily.customer_segment,
    dcgm_daily.customer_name,
    dcgm_daily.region_rollup,
    dcgm_daily.zone,
    dcgm_daily.gen,
    dcgm_daily.product_resolved,
    dcgm_daily.peak_power_unit,
    dcgm_daily.gpu_util_p50,
    dcgm_daily.gpu_util_p95,
    dcgm_daily.gpu_util_p99,
    dcgm_daily.tensor_util_p50,
    dcgm_daily.tensor_util_p95,
    dcgm_daily.tensor_util_p99,
    dcgm_daily.chip_power_fleet_p50,
    dcgm_daily.chip_power_fleet_p95,
    dcgm_daily.chip_power_fleet_p99,
    dcgm_daily.redfish_power_fleet_p50,
    dcgm_daily.redfish_power_fleet_p95,
    dcgm_daily.redfish_power_fleet_p99,
    dcgm_daily.dram_active_p50,
    dcgm_daily.dram_active_p95,
    dcgm_daily.dram_active_p99,
    dcgm_daily.mem_copy_util_p50,
    dcgm_daily.mem_copy_util_p95,
    dcgm_daily.mem_copy_util_p99,
    dcgm_daily.vram_usage_fleet_p50,
    dcgm_daily.vram_usage_fleet_p95,
    dcgm_daily.vram_usage_fleet_p99,
    dcgm_daily.sm_active_p50,
    dcgm_daily.sm_active_p95,
    dcgm_daily.sm_active_p99,
    dcgm_daily.sm_clock_p50,
    dcgm_daily.sm_clock_p95,
    dcgm_daily.sm_clock_p99,
    dcgm_daily.sm_occupancy_p50,
    dcgm_daily.sm_occupancy_p95,
    dcgm_daily.sm_occupancy_p99,
    dcgm_daily.memory_boundness_index_p50,
    dcgm_daily.memory_boundness_index_p95,
    dcgm_daily.memory_boundness_index_p99,
    dcgm_daily.TFLOPS_per_watt_efficiency_p50,
    dcgm_daily.TFLOPS_per_watt_efficiency_p95,
    dcgm_daily.TFLOPS_per_watt_efficiency_p99,
    dcgm_daily.compute_occupancy_index_p50,
    dcgm_daily.compute_occupancy_index_p95,
    dcgm_daily.compute_occupancy_index_p99,
    dcgm_daily.tensor_util_dram_index_p50,
    dcgm_daily.tensor_util_dram_index_p95,
    dcgm_daily.tensor_util_dram_index_p99,
    dcgm_daily.tensor_tflops_sm_fleet_p50,
    dcgm_daily.tensor_tflops_sm_fleet_p95,
    dcgm_daily.tensor_tflops_sm_fleet_p99,
    dcgm_daily.tensor_to_gpu_ratio_p50,
    dcgm_daily.tensor_to_gpu_ratio_p95,
    dcgm_daily.tensor_to_gpu_ratio_p99,
    dcgm_daily.memory_intensity_p50,
    dcgm_daily.memory_intensity_p95,
    dcgm_daily.memory_intensity_p99,
    dcgm_daily.power_pct_max_p50,
    dcgm_daily.power_pct_max_p95,
    dcgm_daily.power_pct_max_p99,
    dcgm_daily.compute_to_memory_ratio_p50,
    dcgm_daily.compute_to_memory_ratio_p95,
    dcgm_daily.compute_to_memory_ratio_p99,
    dcgm_daily.compute_pct_max_p50,
    dcgm_daily.compute_pct_max_p95,
    dcgm_daily.compute_pct_max_p99,
    dcgm_daily.node_count_daily_avg,
    dcgm_daily.tensor_tflops_fleet_avg,
    dcgm_daily.tensor_tflops_fleet_p50,
    dcgm_daily.tensor_tflops_fleet_p95,
    dcgm_daily.tensor_tflops_fleet_p99,
    dcgm_daily.tflops_node_avg_p50,
    dcgm_daily.tflops_node_avg_p95,
    dcgm_daily.tflops_node_avg_p99
FROM sandbox.sandbox_finance.dcgm_metrics_summary_imputed_daily_v2 AS dcgm_daily;
'''

In [8]:
ness.sql(sql_test)

DataFrame[]

In [6]:
# Check available columns in the source table
check_sql = """
DESCRIBE TABLE sandbox.sandbox_finance.dcgm_metrics_summary_imputed_daily_v2
"""

try:
    df_schema = ness.sql(check_sql)
    print("Available columns in source table:")
    df_schema.show(200, truncate=False)
except Exception as e:
    print(f"Error describing table: {e}")
    print("\nTrying alternative approach - checking with SELECT * LIMIT 1:")
    check_sql_alt = """
    SELECT * FROM sandbox.sandbox_finance.dcgm_metrics_summary_imputed_daily_v2 LIMIT 1
    """
    df_sample = ness.sql(check_sql_alt)
    print("Columns available:")
    for col in df_sample.columns:
        print(f"  - {col}")

Available columns in source table:
+------------------------------+---------+-------+
|col_name                      |data_type|comment|
+------------------------------+---------+-------+
|day                           |timestamp|NULL   |
|region                        |string   |NULL   |
|region_rollup                 |string   |NULL   |
|zone                          |string   |NULL   |
|gen                           |string   |NULL   |
|is_training                   |string   |NULL   |
|is_multinode                  |string   |NULL   |
|product_resolved              |string   |NULL   |
|product_segment               |string   |NULL   |
|gpu_count_expected            |int      |NULL   |
|customer_segment              |string   |NULL   |
|customer_name                 |string   |NULL   |
|peak_power_unit               |double   |NULL   |
|gpu_util_p50                  |double   |NULL   |
|gpu_util_p95                  |double   |NULL   |
|gpu_util_p99                  |double   |NULL 

In [ ]:
# Create view with only existing columns
# First, let's create a simpler version that should work

sql_fixed = '''
CREATE OR REPLACE VIEW sandbox.sandbox_jbok.v_dcgm_metrics_summary_imputed_daily
AS
SELECT * FROM sandbox.sandbox_finance.dcgm_metrics_summary_imputed_daily_v2
'''

print("Creating view with all columns from source table...")
try:
    ness.sql(sql_fixed)
    print("✓ View created successfully!")
except Exception as e:
    print(f"Error creating view: {e}")